# 06a: Generate Decoder Training Data (MiniT5)

**Runtime: A100** (Mistral-7B needs the VRAM)

Uses Mistral-7B-Instruct as teacher to generate (encoder_input, decoder_target) pairs.
MiniT5 will be trained from scratch on this data — zero pretrained knowledge.

**Pipeline:**
1. Generate 10 questions per fact via Mistral-7B (~600 base Q&A)
2. Generate 200 cross-domain pairs (multi-fact questions)
3. Generate 100 no-knowledge pairs ("I don't know")
4. Augment to ~60K examples
5. Download `decoder_training.jsonl` — upload it in notebook 06b

**Time:** ~1-2 hrs on A100 | **Cost:** $0

In [ ]:
!pip install -q sentence-transformers
!nvidia-smi | head -4

In [ ]:
!git clone https://github.com/ibrahimm8x/HCL.git /content/HLC

import sys
sys.path.insert(0, '/content/HLC')

In [ ]:
from pathlib import Path
from hlc.config import Config
from hlc.decoder_training import DecoderTrainer
from experiments.seed_basic import BASIC_FACTS

config = Config(
    data_dir=Path('/content/HLC/data'),
    columns_dir=Path('/content/HLC/data/columns'),
    faiss_dir=Path('/content/HLC/data/faiss_index'),
    decoder_model_path=Path('/content/HLC/data/decoder_model'),
)
trainer = DecoderTrainer(config)

DATA_DIR = Path('/content/HLC/data')
DATA_DIR.mkdir(parents=True, exist_ok=True)
BASE_DATASET = DATA_DIR / 'decoder_base.jsonl'
TRAINING_DATA = DATA_DIR / 'decoder_training.jsonl'

knowledge_facts = [f for f in BASIC_FACTS if not f.startswith('STRATEGY:')]
print(f'Knowledge facts: {len(knowledge_facts)}')
print(f'Strategy facts: {len(BASIC_FACTS) - len(knowledge_facts)}')
print(f'Teacher model: {config.lm_model_name}')

## Step 1: Generate base dataset via Mistral-7B (~1-2 hrs)

This generates:
- ~600 Q&A pairs (10 questions per fact)
- ~200 cross-domain pairs (multi-fact synthesis)
- ~100 no-knowledge pairs ("I don't know")

Total: ~900 base examples

In [ ]:
base_examples = trainer.generate_base_dataset(
    facts=BASIC_FACTS,
    output_path=BASE_DATASET,
    verbose=True,
)
print(f'\nBase dataset: {len(base_examples)} examples')
print(f'  With knowledge: {sum(1 for ex in base_examples if ex.knowledge)}')
print(f'  No knowledge: {sum(1 for ex in base_examples if not ex.knowledge)}')

In [ ]:
# Preview base examples
for i, ex in enumerate(base_examples[:5]):
    print(f'--- Example {i+1} ---')
    print(f'Q: {ex.query}')
    print(f'K: {ex.knowledge[0][:80]}...' if ex.knowledge else 'K: (none)')
    print(f'R: {ex.response}')
    print()

# Preview a no-knowledge example
no_k = [ex for ex in base_examples if not ex.knowledge]
if no_k:
    print('--- No-knowledge example ---')
    print(f'Q: {no_k[0].query}')
    print(f'K: (none)')
    print(f'R: {no_k[0].response}')

## Step 2: Augment to ~60K (instant, no GPU)

In [ ]:
augmented = trainer.augment_dataset(base_examples, target_size=60000)
trainer.prepare_training_data(augmented, TRAINING_DATA)

print(f'Augmented: {len(augmented)} examples')
print(f'  With knowledge: {sum(1 for ex in augmented if ex.knowledge)}')
print(f'  No knowledge: {sum(1 for ex in augmented if not ex.knowledge)}')
print(f'  No-knowledge ratio: {sum(1 for ex in augmented if not ex.knowledge) / len(augmented):.1%}')
print(f'Saved to: {TRAINING_DATA}')
print(f'File size: {TRAINING_DATA.stat().st_size / 1024 / 1024:.1f} MB')

In [ ]:
# Preview formatted training examples
import json

with open(TRAINING_DATA) as f:
    lines = [json.loads(line) for line in f]

print('=== Sample encoder_input ===')
print(lines[0]['encoder_input'][:200])
print()
print('=== Sample decoder_target ===')
print(lines[0]['decoder_target'])
print()

# Show a no-knowledge example
no_k_lines = [l for l in lines if '<noknowledge>' in l['encoder_input']]
if no_k_lines:
    print('=== No-knowledge encoder_input ===')
    print(no_k_lines[0]['encoder_input'])
    print()
    print('=== No-knowledge decoder_target ===')
    print(no_k_lines[0]['decoder_target'])

## Step 3: Download training data

Download `decoder_training.jsonl` before switching runtime.
Upload it in notebook 06b (which can run on T4 free tier).

In [ ]:
from google.colab import files

files.download(str(TRAINING_DATA))
print(f'Downloaded: decoder_training.jsonl ({TRAINING_DATA.stat().st_size / 1024 / 1024:.1f} MB)')